# AI Content Development - ASML Graduation Project

**Student:** Neil Ross Daniel (221270)  
**Client:** ASML  
**Supervisors:** Aditi Raool, Christos Lazaridis (ASML) | Karna Rewatkar (BUas)  
**Project Duration:** 12 February 2026 - 19 June 2026 (18 weeks)

---

## Project Overview

This notebook implements an **AI-powered micro-learning system** that automatically retrieves and assembles short (1-2 minute) troubleshooting micro-clips from existing training videos. The system enables technicians to quickly access the exact steps needed to fix specific machine issues.

### Incremental Goals
1. **Goal 1:** Extract and structure spoken maintenance instructions using ASR and LLMs (without timestamps)
2. **Goal 2:** Generate accurate timestamps for relevant instructional segments
3. **Goal 3:** Automatically identify relevant videos/segments for maintenance queries
4. **Goal 4:** Assemble safety-compliant micro-clips into coherent troubleshooting guides

---
# Phase 1: Project Setup and Configuration
*Weeks 1-2: Project Initiation and Problem Definition*

---

## 1.1 Environment Setup and Dependencies

In [7]:
# Install required packages (uncomment as needed)
# !pip install openai-whisper
# !pip install transformers
# !pip install torch torchvision torchaudio
# !pip install langchain langchain-community
# !pip install chromadb faiss-cpu
# !pip install sentence-transformers
# !pip install moviepy
# !pip install ffmpeg-python
# !pip install pandas numpy matplotlib seaborn
# !pip install pydub
# !pip install librosa
# !pip install opencv-python
# !pip install gradio  # For prototype interface

In [2]:
# Core imports
import os
import json
import logging
from pathlib import Path
from datetime import datetime, timedelta
from typing import List, Dict, Tuple, Optional, Any
from dataclasses import dataclass, field
import warnings
warnings.filterwarnings('ignore')

# Data processing
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

print("Core imports loaded successfully!")

Core imports loaded successfully!


In [3]:
# ML/AI imports (load when needed to save memory)
def load_ml_dependencies():
    """Lazy load ML dependencies."""
    global torch, whisper, transformers
    import torch
    import whisper
    import transformers
    from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
    
    
    print(f"PyTorch version: {torch.__version__}")
    print(f"CUDA available: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
    return torch, whisper, transformers

torch, whisper, transformers = load_ml_dependencies()

PyTorch version: 2.10.0+cpu
CUDA available: False


## 1.2 Project Configuration

In [4]:
@dataclass
class ProjectConfig:
    """Central configuration for the AI Content Development project."""
    
    # Project metadata
    project_name: str = "ASML AI Content Development"
    version: str = "0.1.0"
    
    # Paths
    data_dir: Path = Path("./Data/Test")
    raw_videos_dir: Path = Path("./Data/raw_videos")
    processed_dir: Path = Path("./Data/processed")
    transcripts_dir: Path = Path("./Data/transcripts")
    embeddings_dir: Path = Path("./Data/embeddings")
    output_dir: Path = Path("./output")
    models_dir: Path = Path("./models")
    logs_dir: Path = Path("./logs")
    
    # ASR Configuration
    whisper_model: str = "large-v3"  # Options: tiny, base, small, medium, large, large-v2, large-v3
    whisper_language: str = "en"
    
    # LLM Configuration
    llm_model: str = "gpt-4"  # Or local model path
    llm_temperature: float = 0.1
    llm_max_tokens: int = 4096
    
    # Embedding Configuration
    embedding_model: str = "sentence-transformers/all-MiniLM-L6-v2"
    
    # Video Processing
    target_clip_duration: Tuple[int, int] = (60, 120)  # 1-2 minutes in seconds
    video_fps: int = 30
    
    # Retrieval Configuration
    top_k_retrieval: int = 5
    similarity_threshold: float = 0.7
    
    def __post_init__(self):
        """Create necessary directories."""
        for path_attr in ['data_dir', 'raw_videos_dir', 'processed_dir', 
                          'transcripts_dir', 'embeddings_dir', 'output_dir', 
                          'models_dir', 'logs_dir']:
            path = getattr(self, path_attr)
            path.mkdir(parents=True, exist_ok=True)
            
    def save(self, filepath: str = "config.json"):
        """Save configuration to JSON."""
        config_dict = {
            k: str(v) if isinstance(v, Path) else v 
            for k, v in self.__dict__.items()
        }
        with open(filepath, 'w') as f:
            json.dump(config_dict, f, indent=2)
            
    @classmethod
    def load(cls, filepath: str = "config.json"):
        """Load configuration from JSON."""
        with open(filepath, 'r') as f:
            config_dict = json.load(f)
        return cls(**config_dict)

# Initialize configuration
config = ProjectConfig()
config.save()
print(f"Project '{config.project_name}' initialized successfully!")
print(f"Data directory: {config.data_dir.absolute()}")

Project 'ASML AI Content Development' initialized successfully!
Data directory: c:\Users\nedaniel\Documents\CS\2025-26-graduation-neildaniel221270\Data\Test


## 1.3 Data Structures

In [5]:
@dataclass
class TranscriptSegment:
    """Represents a segment of transcribed audio with timestamps."""
    text: str
    start_time: float  # seconds
    end_time: float    # seconds
    confidence: float = 1.0
    speaker: Optional[str] = None
    
    @property
    def duration(self) -> float:
        return self.end_time - self.start_time
    
    def to_dict(self) -> Dict:
        return {
            'text': self.text,
            'start_time': self.start_time,
            'end_time': self.end_time,
            'confidence': self.confidence,
            'speaker': self.speaker
        }

@dataclass 
class VideoMetadata:
    """Metadata for a training video."""
    video_id: str
    title: str
    filepath: Path
    duration: float  # seconds
    topics: List[str] = field(default_factory=list)
    machine_types: List[str] = field(default_factory=list)
    error_codes: List[str] = field(default_factory=list)
    safety_notes: List[str] = field(default_factory=list)
    created_date: Optional[datetime] = None
    
@dataclass
class InstructionalSegment:
    """A semantically meaningful instructional segment."""
    segment_id: str
    video_id: str
    title: str
    summary: str
    start_time: float
    end_time: float
    transcript_segments: List[TranscriptSegment]
    steps: List[str] = field(default_factory=list)
    tools_required: List[str] = field(default_factory=list)
    safety_warnings: List[str] = field(default_factory=list)
    relevance_score: float = 0.0
    embedding: Optional[np.ndarray] = None
    
@dataclass
class MicroClip:
    """A generated micro-clip for troubleshooting."""
    clip_id: str
    query: str
    source_segments: List[InstructionalSegment]
    output_path: Optional[Path] = None
    total_duration: float = 0.0
    captions: List[Dict] = field(default_factory=list)
    safety_validated: bool = False
    validation_notes: str = ""
    created_at: datetime = field(default_factory=datetime.now)

print("Data structures defined successfully!")

Data structures defined successfully!


---
# Phase 2: Literature Review & Requirements
*Weeks 3-4: Literature Review and Requirements Analysis*

---

## 2.1 Key Research Areas

Document your literature review findings here:

### Microlearning Effectiveness
- Moore et al. (2024): Mobile-based microlearning in adult learner contexts
- Corbeil & Corbeil (2024): Designing microlearning for how people learn

### Video Summarization with LLMs/VLMs
- Lee et al. (CVPR 2025): Video Summarization with Large Language Models (LLMVS)
- Tang et al. (2024-2025): Video Understanding with LLMs Survey (Vid-LLM)

### RAG for Enterprise Video
- Arefeen et al. (CVPRW 2024): ViTA - Efficient Video-to-Text for RAG
- Dela Rosa (SIGIR 2024): Video-Enriched RAG using aligned captions

### ASR Accuracy
- Speechmatics: ASR Accuracy Benchmarking & WER guidance
- 3Play Media (2024): State of ASR report

### Maintenance Knowledge Graphs
- Chen et al. (2025): Knowledge-Graph-Driven Fault Diagnosis
- Wu et al. (2025): Knowledge graph-embedded LLM for machine fault identification

In [ ]:
# Literature tracking DataFrame
literature_review = pd.DataFrame({
    'paper': [
        'Moore et al. (2024)',
        'Lee et al. (CVPR 2025)',
        'Arefeen et al. (CVPRW 2024)',
        'Speechmatics ASR Benchmarking',
        'Chen et al. (2025)'
    ],
    'topic': [
        'Microlearning',
        'Video Summarization',
        'Video RAG',
        'ASR Accuracy',
        'Knowledge Graphs'
    ],
    'key_findings': [
        'Microlearning improves retention in workplace settings',
        'LLMs enable semantic scene selection for user-query-aware summaries',
        'VLM-based video-to-text improves retrieval accuracy',
        'Modern ASR achieves <5% WER on clean audio',
        'KG-driven fault diagnosis improves troubleshooting efficiency'
    ],
    'relevance_to_project': [
        'Validates micro-clip approach for technician training',
        'Core technique for segment selection',
        'Architecture for video retrieval system',
        'Baseline expectations for transcription quality',
        'Potential enhancement for error-code mapping'
    ],
    'status': ['Reviewed', 'Reviewed', 'Reviewed', 'Reviewed', 'Reviewed']
})

literature_review

,paper,topic,key_findings,relevance_to_project,status
0,Moore et al. (2024),Microlearning,Microlearning improves retention in workplace ...,Validates micro-clip approach for technician t...,Reviewed
1,Lee et al. (CVPR 2025),Video Summarization,LLMs enable semantic scene selection for user-...,Core technique for segment selection,Reviewed
2,Arefeen et al. (CVPRW 2024),Video RAG,VLM-based video-to-text improves retrieval acc...,Architecture for video retrieval system,Reviewed
3,Speechmatics ASR Benchmarking,ASR Accuracy,Modern ASR achieves <5% WER on clean audio,Baseline expectations for transcription quality,Reviewed
4,Chen et al. (2025),Knowledge Graphs,KG-driven fault diagnosis improves troubleshoo...,Potential enhancement for error-code mapping,To Review


## 2.2 Requirements Specification

In [9]:
# Functional Requirements
functional_requirements = {
    'FR-01': {
        'description': 'System shall transcribe training videos with >95% accuracy',
        'priority': 'Must Have',
        'goal': 1,
        'acceptance_criteria': 'WER < 5% on test set'
    },
    'FR-02': {
        'description': 'System shall structure transcripts into semantic segments',
        'priority': 'Must Have',
        'goal': 1,
        'acceptance_criteria': 'Coherent segments identified by SME review'
    },
    'FR-03': {
        'description': 'System shall generate accurate timestamps (±2 seconds)',
        'priority': 'Must Have',
        'goal': 2,
        'acceptance_criteria': 'Timestamp accuracy >90% on validation set'
    },
    'FR-04': {
        'description': 'System shall retrieve relevant segments based on error queries',
        'priority': 'Must Have',
        'goal': 3,
        'acceptance_criteria': 'Precision@5 > 0.8'
    },
    'FR-05': {
        'description': 'System shall assemble micro-clips of 1-2 minutes',
        'priority': 'Must Have',
        'goal': 4,
        'acceptance_criteria': 'Generated clips within duration range'
    },
    'FR-06': {
        'description': 'System shall include safety warnings in generated clips',
        'priority': 'Must Have',
        'goal': 4,
        'acceptance_criteria': 'Safety review approval'
    }
}

# Non-Functional Requirements
non_functional_requirements = {
    'NFR-01': {
        'description': 'System shall process a 1-hour video within 30 minutes',
        'category': 'Performance'
    },
    'NFR-02': {
        'description': 'System shall be GDPR compliant',
        'category': 'Legal/Compliance'
    },
    'NFR-03': {
        'description': 'System shall provide reproducible results',
        'category': 'Reliability'
    },
    'NFR-04': {
        'description': 'System shall be scalable to 1000+ videos',
        'category': 'Scalability'
    }
}

pd.DataFrame(functional_requirements).T

,description,priority,goal,acceptance_criteria
FR-01,System shall transcribe training videos with >...,Must Have,1,WER < 5% on test set
FR-02,System shall structure transcripts into semant...,Must Have,1,Coherent segments identified by SME review
FR-03,System shall generate accurate timestamps (±2 ...,Must Have,2,Timestamp accuracy >90% on validation set
FR-04,System shall retrieve relevant segments based ...,Must Have,3,Precision@5 > 0.8
FR-05,System shall assemble micro-clips of 1-2 minutes,Must Have,4,Generated clips within duration range
FR-06,System shall include safety warnings in genera...,Must Have,4,Safety review approval


## 2.3 Evaluation Framework

In [10]:
# Define evaluation metrics for each goal
evaluation_metrics = {
    'Goal 1 - ASR & Structuring': {
        'metrics': ['Word Error Rate (WER)', 'Character Error Rate (CER)', 
                   'Segment Coherence Score (manual)', 'Processing Time'],
        'targets': ['< 5%', '< 3%', '> 4/5 SME rating', '< 0.5x video duration']
    },
    'Goal 2 - Timestamp Generation': {
        'metrics': ['Timestamp Accuracy', 'Boundary Detection F1', 
                   'Segment Duration Variance'],
        'targets': ['±2 seconds', '> 0.85', '< 10 seconds std']
    },
    'Goal 3 - Retrieval': {
        'metrics': ['Precision@K', 'Recall@K', 'Mean Reciprocal Rank (MRR)', 
                   'Normalized Discounted Cumulative Gain (NDCG)'],
        'targets': ['> 0.8', '> 0.7', '> 0.75', '> 0.8']
    },
    'Goal 4 - Micro-Clip Assembly': {
        'metrics': ['Clip Coherence Score', 'Safety Compliance Rate', 
                   'User Satisfaction Score', 'Task Completion Rate'],
        'targets': ['> 4/5', '100%', '> 4/5', '> 80%']
    }
}

for goal, data in evaluation_metrics.items():
    print(f"\n{goal}")
    print("-" * 50)
    for metric, target in zip(data['metrics'], data['targets']):
        print(f"  {metric}: {target}")


Goal 1 - ASR & Structuring
--------------------------------------------------
  Word Error Rate (WER): < 5%
  Character Error Rate (CER): < 3%
  Segment Coherence Score (manual): > 4/5 SME rating
  Processing Time: < 0.5x video duration

Goal 2 - Timestamp Generation
--------------------------------------------------
  Timestamp Accuracy: ±2 seconds
  Boundary Detection F1: > 0.85
  Segment Duration Variance: < 10 seconds std

Goal 3 - Retrieval
--------------------------------------------------
  Precision@K: > 0.8
  Recall@K: > 0.7
  Mean Reciprocal Rank (MRR): > 0.75
  Normalized Discounted Cumulative Gain (NDCG): > 0.8

Goal 4 - Micro-Clip Assembly
--------------------------------------------------
  Clip Coherence Score: > 4/5
  Safety Compliance Rate: 100%
  User Satisfaction Score: > 4/5
  Task Completion Rate: > 80%


---
# Phase 3: Data Exploration & Baseline Pipeline (Goal 1)
*Weeks 5-7: ASR Transcription and LLM Structuring*

---

## 3.1 Data Loading and Exploration

In [6]:
class VideoDataLoader:
    """Handles loading and basic processing of training videos."""
    
    def __init__(self, config: ProjectConfig):
        self.config = config
        self.supported_formats = ['.mp4', '.avi', '.mov', '.mkv', '.webm']
        
    def scan_videos(self) -> List[VideoMetadata]:
        """Scan the raw videos directory and catalog all videos."""
        videos = []
        
        for video_path in self.config.raw_videos_dir.glob('**/*'):
            if video_path.suffix.lower() in self.supported_formats:
                metadata = self._extract_metadata(video_path)
                videos.append(metadata)
                
        logger.info(f"Found {len(videos)} videos in {self.config.raw_videos_dir}")
        return videos
    
    def _extract_metadata(self, video_path: Path) -> VideoMetadata:
        """Extract metadata from a video file."""
        # TODO: Implement actual metadata extraction using moviepy/ffprobe
        return VideoMetadata(
            video_id=video_path.stem,
            title=video_path.stem.replace('_', ' ').title(),
            filepath=video_path,
            duration=0.0,  # Will be populated by actual extraction
            topics=[],
            machine_types=[],
            error_codes=[]
        )
    
    def get_video_stats(self, videos: List[VideoMetadata]) -> pd.DataFrame:
        """Generate statistics about the video corpus."""
        stats = []
        for v in videos:
            stats.append({
                'video_id': v.video_id,
                'title': v.title,
                'duration_min': v.duration / 60,
                'num_topics': len(v.topics),
                'num_error_codes': len(v.error_codes)
            })
        return pd.DataFrame(stats)

# Initialize data loader
data_loader = VideoDataLoader(config)

# Scan for videos (will be empty until data is added)
videos = data_loader.scan_videos()
print(f"Loaded {len(videos)} videos")

2026-03-04 10:29:55,071 - INFO - Found 4 videos in Data\raw_videos


Loaded 4 videos


## 3.2 ASR Pipeline (Whisper)

In [7]:
class ASRPipeline:
    """Automatic Speech Recognition pipeline using Whisper."""
    
    def __init__(self, config: ProjectConfig):
        self.config = config
        self.model = None
        
    def load_model(self):
        """Load the Whisper model."""
        import whisper
        logger.info(f"Loading Whisper model: {self.config.whisper_model}")
        self.model = whisper.load_model(self.config.whisper_model)
        logger.info("Model loaded successfully")
        
    def transcribe(self, video_path: Path, 
                   word_timestamps: bool = True) -> List[TranscriptSegment]:
        """Transcribe a video file and return segments with timestamps."""
        if self.model is None:
            self.load_model()
            
        logger.info(f"Transcribing: {video_path.name}")
        
        # Transcribe with word-level timestamps
        result = self.model.transcribe(
            str(video_path),
            language=self.config.whisper_language,
            word_timestamps=word_timestamps,
            verbose=False
        )
        
        # Convert to TranscriptSegment objects
        segments = []
        for seg in result['segments']:
            segments.append(TranscriptSegment(
                text=seg['text'].strip(),
                start_time=seg['start'],
                end_time=seg['end'],
                confidence=seg.get('confidence', 1.0)
            ))
            
        logger.info(f"Transcribed {len(segments)} segments")
        return segments
    
    def save_transcript(self, segments: List[TranscriptSegment], 
                        video_id: str) -> Path:
        """Save transcript to JSON file."""
        output_path = self.config.transcripts_dir / f"{video_id}_transcript.json"
        
        data = {
            'video_id': video_id,
            'created_at': datetime.now().isoformat(),
            'num_segments': len(segments),
            'segments': [seg.to_dict() for seg in segments]
        }
        
        with open(output_path, 'w') as f:
            json.dump(data, f, indent=2)
            
        logger.info(f"Saved transcript to {output_path}")
        return output_path

# Initialize ASR pipeline
asr_pipeline = ASRPipeline(config)
print("ASR Pipeline initialized (model not loaded yet)")

ASR Pipeline initialized (model not loaded yet)


In [ ]:
# Transcribe a video
video_path = config.raw_videos_dir / "LMOS Introduction to the process.mp4"
if video_path.exists():
    segments = asr_pipeline.transcribe(video_path)
    asr_pipeline.save_transcript(segments, "LMOS Introduction to the process")
    
    # Display first few segments
    for i, seg in enumerate(segments[:5]):
        print(f"[{seg.start_time:.2f}s - {seg.end_time:.2f}s] {seg.text}")

2026-03-03 10:41:11,469 - INFO - Loading Whisper model: large-v3
100%|█████████████████████████████████████| 2.88G/2.88G [00:50<00:00, 61.0MiB/s]
2026-03-03 10:42:21,047 - INFO - Model loaded successfully
2026-03-03 10:42:21,049 - INFO - Transcribing: LMOS Introduction to the process.mp4
100%|██████████| 15876/15876 [09:05<00:00, 29.10frames/s]
2026-03-03 10:51:28,310 - INFO - Transcribed 25 segments
2026-03-03 10:51:28,328 - INFO - Saved transcript to Data\transcripts\LMOS Introduction to the process_transcript.json


[8.58s - 14.92s] Elmos is the ASML lean maintenance operating system for the maintenance of our customers installed base.
[15.42s - 21.52s] It is a standardized way of working based on best known methods from many of the ASML sites.
[22.40s - 28.94s] Elmos will drive the predictable and high availability of our customer systems and drives the continuous improvement.
[28.94s - 35.42s] The systems health of the installed base is closely monitored using the collective knowledge of our organization.
[36.18s - 42.48s] The ambition is to capture future failures, initiate timely maintenance and avoid an unscheduled down.


## 3.3 LLM-Based Structuring and Summarization

In [8]:
import json
from pathlib import Path
from typing import List

def load_transcript_json(transcript_path: Path) -> List[TranscriptSegment]:
    with open(transcript_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    segments = []
    for seg in data["segments"]:
        segments.append(
            TranscriptSegment(
                text=seg["text"],
                start_time=float(seg["start_time"]),
                end_time=float(seg["end_time"]),
                confidence=float(seg.get("confidence", 1.0)),
                speaker=seg.get("speaker")
            )
        )
    return segments

In [9]:
import json
import uuid
from dataclasses import asdict
from typing import List, Dict, Optional, Tuple
from datetime import datetime

from pydantic import BaseModel, Field, ValidationError

# LangChain (modern)
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate


class LLMStructuredSegment(BaseModel):
    title: str = Field(..., description="Concise instructional segment title")
    summary: str = Field(..., description="2-3 sentence summary")
    text: List[str] = Field(default_factory=list, description="Key lines or bullets from transcript")
    tools: List[str] = Field(default_factory=list, description="Tools/equipment mentioned")
    safety_warnings: List[str] = Field(default_factory=list, description="Safety warnings/precautions")
    start_time: float = Field(..., ge=0)
    end_time: float = Field(..., ge=0)


class LLMStructuredResponse(BaseModel):
    segments: List[LLMStructuredSegment] = Field(default_factory=list)


class TranscriptStructurer:
    """Uses LLMs to structure and summarize transcripts."""

    STRUCTURING_SYSTEM = (
        "You are an expert at analyzing machine maintenance training transcripts. "
        "Your job is to group transcript lines into logical instructional segments.\n"
        "Rules:\n"
        "- Segments MUST be chronological and non-overlapping.\n"
        "- start_time/end_time MUST match timestamps present in the transcript.\n"
        "- Keep segments coherent (one task/procedure per segment).\n"
        "- Extract tools and safety warnings if mentioned.\n"
        "- Output MUST be valid JSON matching the provided schema.\n"
    )

    STRUCTURING_USER_TEMPLATE = """Transcript lines (timestamped):
{transcript}

Return JSON with:
{schema_hint}
"""

    SUMMARY_SYSTEM = (
        "You are an expert technical summarizer for maintenance training videos. "
        "Write clear, factual summaries."
    )

    SUMMARY_USER_TEMPLATE = """Create:
1) A high-level summary (6-10 bullets)
2) A list of key tools/equipment mentioned (unique)
3) A list of safety warnings (unique)
4) A short 'What this video enables you to do' paragraph (2-3 sentences)

Transcript excerpt:
{text}
"""

    def __init__(self, config: ProjectConfig):
        self.config = config
        self.llm: Optional[ChatOpenAI] = None

    def initialize_llm(self, api_key: Optional[str] = None):
        """
        Initialize OpenAI chat model.
        You can also rely on env var OPENAI_API_KEY instead of passing api_key.
        """
        self.llm = ChatOpenAI(
            model=self.config.llm_model,
            temperature=self.config.llm_temperature,
            max_tokens=self.config.llm_max_tokens,
            api_key=api_key
        )
        logger.info(f"LLM initialized: {self.config.llm_model}")

    # ---------- Utilities ----------

    def _chunk_segments(
        self,
        segments: List[TranscriptSegment],
        chunk_seconds: int = 240,        # 4 minutes per chunk
        overlap_seconds: int = 20
    ) -> List[List[TranscriptSegment]]:
        """Chunk transcript by time window to control prompt size."""
        if not segments:
            return []

        chunks = []
        current = []
        chunk_start = segments[0].start_time
        chunk_end = chunk_start + chunk_seconds

        for seg in segments:
            if seg.end_time <= chunk_end:
                current.append(seg)
            else:
                if current:
                    chunks.append(current)

                # start new chunk with overlap
                overlap_start = max(chunk_start, chunk_end - overlap_seconds)
                current = [s for s in segments if s.start_time >= overlap_start and s.end_time <= seg.end_time]
                # add current seg if missing
                if seg not in current:
                    current.append(seg)

                chunk_start = current[0].start_time
                chunk_end = chunk_start + chunk_seconds

        if current:
            chunks.append(current)

        return chunks

    def _format_transcript_for_prompt(self, segs: List[TranscriptSegment]) -> str:
        return "\n".join([f"[{s.start_time:.2f}s - {s.end_time:.2f}s] {s.text}" for s in segs])

    def _clean_structured_segments(
        self,
        llm_segments: List[LLMStructuredSegment],
        min_duration: float = 5.0
    ) -> List[LLMStructuredSegment]:
        """Enforce monotonic, non-overlapping segments and drop tiny/invalid ones."""
        cleaned = []
        last_end = 0.0

        # Sort by start_time
        llm_segments = sorted(llm_segments, key=lambda x: x.start_time)

        for s in llm_segments:
            # Basic validity
            if s.end_time <= s.start_time:
                continue
            if (s.end_time - s.start_time) < min_duration:
                continue

            # Enforce non-overlap
            if s.start_time < last_end:
                s.start_time = last_end

            if s.end_time <= s.start_time:
                continue

            cleaned.append(s)
            last_end = s.end_time

        return cleaned

    def _map_to_instructional_segments(
        self,
        video_id: str,
        all_transcript_segments: List[TranscriptSegment],
        llm_segments: List[LLMStructuredSegment],
    ) -> List[InstructionalSegment]:
        """Create InstructionalSegment objects and attach the underlying transcript lines."""
        results = []

        for idx, seg in enumerate(llm_segments, start=1):
            # Select transcript segments that fall within [start_time, end_time]
            attached = [
                t for t in all_transcript_segments
                if t.start_time >= seg.start_time and t.end_time <= seg.end_time
            ]

            results.append(
                InstructionalSegment(
                    segment_id=f"{video_id}_seg_{idx:03d}",
                    video_id=video_id,
                    title=seg.title,
                    summary=seg.summary,
                    start_time=seg.start_time,
                    end_time=seg.end_time,
                    transcript_segments=attached,
                    steps=seg.text,  # optional: treat seg.text as steps/bullets; you can refine later
                    tools_required=seg.tools,
                    safety_warnings=seg.safety_warnings,
                    relevance_score=0.0,
                    embedding=None
                )
            )

        return results

    # ---------- Core: Structuring ----------

    def structure_transcript(
        self,
        video_id: str,
        segments: List[TranscriptSegment],
        chunk_seconds: int = 240
    ) -> List[InstructionalSegment]:
        if self.llm is None:
            self.initialize_llm()

        # Build prompt with schema hint
        schema_hint = json.dumps(LLMStructuredResponse.model_json_schema(), indent=2)

        prompt = ChatPromptTemplate.from_messages([
            ("system", self.STRUCTURING_SYSTEM),
            ("user", self.STRUCTURING_USER_TEMPLATE)
        ])

        # Chunk transcript to avoid token overflow
        chunks = self._chunk_segments(segments, chunk_seconds=chunk_seconds)
        logger.info(f"Structuring transcript in {len(chunks)} chunk(s).")

        all_llm_segments: List[LLMStructuredSegment] = []

        for i, chunk in enumerate(chunks, start=1):
            transcript_text = self._format_transcript_for_prompt(chunk)

            chain = prompt | self.llm
            msg = chain.invoke({
                "transcript": transcript_text,
                "schema_hint": schema_hint
            })

            raw = msg.content
            try:
                parsed = LLMStructuredResponse.model_validate_json(raw)
                all_llm_segments.extend(parsed.segments)
                logger.info(f"Chunk {i}/{len(chunks)}: extracted {len(parsed.segments)} segment(s).")
            except ValidationError as e:
                logger.error(f"Chunk {i}: LLM output failed schema validation.\n{e}\nRaw:\n{raw}")
                # Optional: you can add a repair pass here (ask LLM to fix JSON)

        # Clean + merge
        cleaned = self._clean_structured_segments(all_llm_segments)
        logger.info(f"After cleaning: {len(cleaned)} total segment(s).")

        # Convert to your dataclass
        instructional = self._map_to_instructional_segments(video_id, segments, cleaned)
        return instructional

    # ---------- Summarizing ----------

    def generate_summary(self, segments: List[TranscriptSegment]) -> Dict[str, object]:
        """
        Map-reduce style summary.
        Returns a dict with bullets/tools/safety/enablement.
        """
        if self.llm is None:
            self.initialize_llm()

        prompt = ChatPromptTemplate.from_messages([
            ("system", self.SUMMARY_SYSTEM),
            ("user", self.SUMMARY_USER_TEMPLATE)
        ])

        # Chunk by text length rather than time; reuse time chunking for simplicity
        chunks = self._chunk_segments(segments, chunk_seconds=300, overlap_seconds=0)

        partials = []
        for chunk in chunks:
            excerpt = self._format_transcript_for_prompt(chunk)
            chain = prompt | self.llm
            msg = chain.invoke({"text": excerpt})
            partials.append(msg.content)

        # Reduce: summarize the partial summaries
        reduce_prompt = ChatPromptTemplate.from_messages([
            ("system", self.SUMMARY_SYSTEM),
            ("user",
             "Combine and deduplicate the following partial summaries into one clean result.\n\n"
             "Partial summaries:\n{partials}\n\n"
             "Return as JSON with keys: bullets (list of strings), tools (list), safety_warnings (list), enablement (string).")
        ])

        reduce_chain = reduce_prompt | self.llm
        reduced = reduce_chain.invoke({"partials": "\n\n---\n\n".join(partials)}).content

        # Best-effort parse; if it's not JSON, return raw
        try:
            return json.loads(reduced)
        except Exception:
            return {"raw_summary": reduced, "partials": partials}

In [ ]:
from pathlib import Path

video_id = "LMOS Introduction to the process"
transcript_path = "./Data/transcripts_raw/LMOS Introduction to the process_transcript.json"

segments = load_transcript_json(transcript_path)

structurer = TranscriptStructurer(config)
structurer.initialize_llm(api_key=None)  # or pass api_key="..."

instructional_segments = structurer.structure_transcript(video_id, segments)

print(f"Created {len(instructional_segments)} instructional segments.")
for s in instructional_segments[:3]:
    print(f"\n[{s.start_time:.1f}-{s.end_time:.1f}] {s.title}")
    print(s.summary)

summary = structurer.generate_summary(segments)
print("\n=== VIDEO SUMMARY ===")
print(summary)

In [11]:
def save_structured_segments(config: ProjectConfig, video_id: str, segments: List[InstructionalSegment]) -> Path:
    out = f"./Data/transcripts_clean{video_id}_structured.json"
    payload = {
        "video_id": video_id,
        "created_at": datetime.now().isoformat(),
        "num_segments": len(segments),
        "segments": [
            {
                "segment_id": s.segment_id,
                "title": s.title,
                "summary": s.summary,
                "start_time": s.start_time,
                "end_time": s.end_time,
                "tools_required": s.tools_required,
                "safety_warnings": s.safety_warnings,
                "steps": s.steps,
                "transcript_segments": [ts.to_dict() for ts in s.transcript_segments],
            }
            for s in segments
        ]
    }
    with open(out, "w", encoding="utf-8") as f:
        json.dump(payload, f, indent=2)
    return out

# out_path = save_structured_segments(config, video_id, instructional_segments)
# print("Saved structured output to:", out_path)

## 3.4 Goal 1 Evaluation

In [12]:
class ASREvaluator:
    """Evaluate ASR quality using standard metrics."""
    
    @staticmethod
    def word_error_rate(reference: str, hypothesis: str) -> float:
        """
        Calculate Word Error Rate (WER).
        WER = (S + D + I) / N
        where S=substitutions, D=deletions, I=insertions, N=words in reference
        """
        # Simple implementation using edit distance
        ref_words = reference.lower().split()
        hyp_words = hypothesis.lower().split()
        
        # Dynamic programming for edit distance
        d = np.zeros((len(ref_words) + 1, len(hyp_words) + 1))
        
        for i in range(len(ref_words) + 1):
            d[i, 0] = i
        for j in range(len(hyp_words) + 1):
            d[0, j] = j
            
        for i in range(1, len(ref_words) + 1):
            for j in range(1, len(hyp_words) + 1):
                if ref_words[i-1] == hyp_words[j-1]:
                    d[i, j] = d[i-1, j-1]
                else:
                    d[i, j] = min(
                        d[i-1, j] + 1,    # deletion
                        d[i, j-1] + 1,    # insertion
                        d[i-1, j-1] + 1   # substitution
                    )
        
        wer = d[len(ref_words), len(hyp_words)] / len(ref_words) if ref_words else 0
        return wer
    
    @staticmethod
    def evaluate_transcription(reference_file: Path, 
                               hypothesis_segments: List[TranscriptSegment]) -> Dict:
        """Evaluate transcription against reference."""
        # Load reference transcript
        with open(reference_file, 'r') as f:
            reference = f.read()
            
        # Combine hypothesis segments
        hypothesis = " ".join([seg.text for seg in hypothesis_segments])
        
        wer = ASREvaluator.word_error_rate(reference, hypothesis)
        
        return {
            'wer': wer,
            'reference_words': len(reference.split()),
            'hypothesis_words': len(hypothesis.split()),
            'num_segments': len(hypothesis_segments)
        }

# Example evaluation
evaluator = ASREvaluator()

# Test WER calculation
ref = "the quick brown fox jumps over the lazy dog"
hyp = "the quick brown fox jumped over a lazy dog"
wer = evaluator.word_error_rate(ref, hyp)
print(f"Example WER: {wer:.2%}")

Example WER: 22.22%


---
# Phase 4: Temporal Alignment & Semantic Segmentation (Goal 2)
*Weeks 8-10: Timestamp Generation and Segment Boundaries*

---

## 4.1 Timestamp Alignment

In [ ]:
class TimestampAligner:
    """Aligns transcript segments with video timestamps."""
    
    def __init__(self, config: ProjectConfig):
        self.config = config
        
    def refine_timestamps(self, segments: List[TranscriptSegment],
                          video_path: Path) -> List[TranscriptSegment]:
        """
        Refine segment timestamps using audio analysis.
        Uses silence detection and speech activity detection.
        """
        # TODO: Implement audio-based timestamp refinement
        # Options:
        # 1. librosa for audio analysis
        # 2. pyannote-audio for speaker diarization
        # 3. webrtcvad for voice activity detection
        
        logger.info(f"Refining timestamps for {len(segments)} segments")
        return segments
    
    def detect_scene_boundaries(self, video_path: Path) -> List[float]:
        """
        Detect visual scene boundaries in the video.
        Returns list of timestamp boundaries.
        """
        # TODO: Implement scene detection using:
        # 1. OpenCV histogram comparison
        # 2. PySceneDetect
        # 3. Deep learning-based approaches
        
        logger.info(f"Detecting scene boundaries in {video_path.name}")
        return []
    
    def merge_audio_visual_boundaries(self, 
                                      transcript_segments: List[TranscriptSegment],
                                      scene_boundaries: List[float]) -> List[TranscriptSegment]:
        """
        Merge transcript boundaries with visual scene boundaries
        for more accurate segmentation.
        """
        # TODO: Implement boundary merging logic
        return transcript_segments

aligner = TimestampAligner(config)
print("Timestamp Aligner initialized")

## 4.2 Semantic Segmentation

In [ ]:
class SemanticSegmenter:
    """
    Segments transcripts into semantically meaningful units.
    Uses both text-based (LLM) and multimodal (VLM) approaches.
    """
    
    def __init__(self, config: ProjectConfig):
        self.config = config
        self.embedding_model = None
        
    def load_embedding_model(self):
        """Load sentence embedding model."""
        from sentence_transformers import SentenceTransformer
        self.embedding_model = SentenceTransformer(self.config.embedding_model)
        logger.info(f"Loaded embedding model: {self.config.embedding_model}")
        
    def compute_embeddings(self, segments: List[TranscriptSegment]) -> np.ndarray:
        """Compute embeddings for all segments."""
        if self.embedding_model is None:
            self.load_embedding_model()
            
        texts = [seg.text for seg in segments]
        embeddings = self.embedding_model.encode(texts, show_progress_bar=True)
        return embeddings
    
    def find_topic_boundaries(self, embeddings: np.ndarray, 
                              threshold: float = 0.3) -> List[int]:
        """
        Find topic boundaries using embedding similarity.
        Returns indices where topic changes occur.
        """
        from sklearn.metrics.pairwise import cosine_similarity
        
        boundaries = [0]
        
        for i in range(1, len(embeddings)):
            sim = cosine_similarity(
                embeddings[i-1].reshape(1, -1),
                embeddings[i].reshape(1, -1)
            )[0, 0]
            
            if sim < threshold:
                boundaries.append(i)
                
        return boundaries
    
    def create_semantic_segments(self, 
                                 transcript_segments: List[TranscriptSegment],
                                 video_id: str) -> List[InstructionalSegment]:
        """
        Create semantically coherent instructional segments.
        """
        # Compute embeddings
        embeddings = self.compute_embeddings(transcript_segments)
        
        # Find topic boundaries
        boundaries = self.find_topic_boundaries(embeddings)
        boundaries.append(len(transcript_segments))  # Add end boundary
        
        # Create segments
        instructional_segments = []
        
        for i in range(len(boundaries) - 1):
            start_idx = boundaries[i]
            end_idx = boundaries[i + 1]
            
            segment_transcripts = transcript_segments[start_idx:end_idx]
            
            # Combine text for the segment
            combined_text = " ".join([s.text for s in segment_transcripts])
            
            # Compute segment embedding (average of component embeddings)
            segment_embedding = np.mean(embeddings[start_idx:end_idx], axis=0)
            
            instructional_segments.append(InstructionalSegment(
                segment_id=f"{video_id}_seg_{i:03d}",
                video_id=video_id,
                title=f"Segment {i+1}",  # Will be refined by LLM
                summary=combined_text[:200] + "...",  # Will be refined by LLM
                start_time=segment_transcripts[0].start_time,
                end_time=segment_transcripts[-1].end_time,
                transcript_segments=segment_transcripts,
                embedding=segment_embedding
            ))
            
        logger.info(f"Created {len(instructional_segments)} semantic segments")
        return instructional_segments

segmenter = SemanticSegmenter(config)
print("Semantic Segmenter initialized")

## 4.3 Goal 2 Evaluation

In [ ]:
class SegmentationEvaluator:
    """Evaluate segmentation quality."""
    
    @staticmethod
    def timestamp_accuracy(predicted: List[float], 
                           ground_truth: List[float],
                           tolerance: float = 2.0) -> float:
        """
        Calculate timestamp accuracy within tolerance (seconds).
        """
        if not ground_truth:
            return 0.0
            
        correct = 0
        for gt in ground_truth:
            for pred in predicted:
                if abs(pred - gt) <= tolerance:
                    correct += 1
                    break
                    
        return correct / len(ground_truth)
    
    @staticmethod
    def boundary_f1(predicted: List[float], 
                    ground_truth: List[float],
                    tolerance: float = 2.0) -> Dict[str, float]:
        """
        Calculate precision, recall, and F1 for boundary detection.
        """
        if not predicted or not ground_truth:
            return {'precision': 0.0, 'recall': 0.0, 'f1': 0.0}
        
        # True positives: predicted boundaries matching ground truth
        tp = 0
        for pred in predicted:
            for gt in ground_truth:
                if abs(pred - gt) <= tolerance:
                    tp += 1
                    break
        
        precision = tp / len(predicted) if predicted else 0
        recall = tp / len(ground_truth) if ground_truth else 0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
        
        return {'precision': precision, 'recall': recall, 'f1': f1}

# Example evaluation
seg_evaluator = SegmentationEvaluator()

# Test with sample data
pred_boundaries = [0.0, 30.5, 62.0, 95.0, 120.0]
gt_boundaries = [0.0, 32.0, 60.0, 93.0, 122.0]

accuracy = seg_evaluator.timestamp_accuracy(pred_boundaries, gt_boundaries)
f1_metrics = seg_evaluator.boundary_f1(pred_boundaries, gt_boundaries)

print(f"Timestamp Accuracy (±2s): {accuracy:.2%}")
print(f"Boundary F1: {f1_metrics['f1']:.3f}")

---
# Phase 5: Retrieval and Relevance Modeling (Goal 3)
*Weeks 11-13: RAG-based Video Segment Retrieval*

---

## 5.1 Vector Store Setup

In [ ]:
class VideoSegmentVectorStore:
    """
    Vector store for video segment retrieval.
    Uses FAISS or ChromaDB for efficient similarity search.
    """
    
    def __init__(self, config: ProjectConfig):
        self.config = config
        self.index = None
        self.segments: List[InstructionalSegment] = []
        self.embedding_model = None
        
    def initialize(self, use_faiss: bool = True):
        """Initialize the vector store."""
        from sentence_transformers import SentenceTransformer
        
        self.embedding_model = SentenceTransformer(self.config.embedding_model)
        
        if use_faiss:
            import faiss
            # Initialize FAISS index (will be populated when segments are added)
            self.index_type = 'faiss'
            logger.info("Initialized FAISS vector store")
        else:
            import chromadb
            self.client = chromadb.Client()
            self.collection = self.client.create_collection("video_segments")
            self.index_type = 'chroma'
            logger.info("Initialized ChromaDB vector store")
            
    def add_segments(self, segments: List[InstructionalSegment]):
        """Add segments to the vector store."""
        self.segments.extend(segments)
        
        # Compute embeddings if not already present
        texts = []
        for seg in segments:
            # Combine title, summary, and steps for better retrieval
            text = f"{seg.title}. {seg.summary}. Steps: {' '.join(seg.steps)}"
            texts.append(text)
            
        embeddings = self.embedding_model.encode(texts)
        
        if self.index_type == 'faiss':
            import faiss
            if self.index is None:
                dim = embeddings.shape[1]
                self.index = faiss.IndexFlatIP(dim)  # Inner product for cosine similarity
            
            # Normalize for cosine similarity
            faiss.normalize_L2(embeddings)
            self.index.add(embeddings)
        else:
            # ChromaDB
            self.collection.add(
                embeddings=embeddings.tolist(),
                ids=[seg.segment_id for seg in segments],
                metadatas=[{'video_id': seg.video_id, 'title': seg.title} for seg in segments]
            )
            
        logger.info(f"Added {len(segments)} segments to vector store")
        
    def search(self, query: str, top_k: int = None) -> List[Tuple[InstructionalSegment, float]]:
        """Search for relevant segments given a query."""
        if top_k is None:
            top_k = self.config.top_k_retrieval
            
        # Encode query
        query_embedding = self.embedding_model.encode([query])
        
        if self.index_type == 'faiss':
            import faiss
            faiss.normalize_L2(query_embedding)
            scores, indices = self.index.search(query_embedding, top_k)
            
            results = []
            for idx, score in zip(indices[0], scores[0]):
                if idx < len(self.segments):
                    results.append((self.segments[idx], float(score)))
                    
            return results
        else:
            # ChromaDB
            results = self.collection.query(
                query_embeddings=query_embedding.tolist(),
                n_results=top_k
            )
            # Convert results to segment objects
            # TODO: Implement ChromaDB result conversion
            return []
            
    def save(self, filepath: Path):
        """Save vector store to disk."""
        if self.index_type == 'faiss':
            import faiss
            faiss.write_index(self.index, str(filepath / "faiss.index"))
            
        # Save segment metadata
        metadata = [{
            'segment_id': s.segment_id,
            'video_id': s.video_id,
            'title': s.title,
            'summary': s.summary,
            'start_time': s.start_time,
            'end_time': s.end_time
        } for s in self.segments]
        
        with open(filepath / "segments_metadata.json", 'w') as f:
            json.dump(metadata, f, indent=2)
            
        logger.info(f"Saved vector store to {filepath}")

# Initialize vector store
# vector_store = VideoSegmentVectorStore(config)
# vector_store.initialize(use_faiss=True)
print("Vector Store class defined")

## 5.2 RAG Pipeline

In [ ]:
class MaintenanceRAGPipeline:
    """
    RAG pipeline for maintenance troubleshooting queries.
    Combines retrieval with LLM-based relevance ranking and response generation.
    """
    
    RELEVANCE_PROMPT = """You are evaluating video segments for relevance to a maintenance query.

Query: {query}

Segment Title: {title}
Segment Summary: {summary}
Steps: {steps}

Rate the relevance of this segment on a scale of 0-10, where:
- 10: Directly addresses the query with step-by-step instructions
- 7-9: Highly relevant, contains useful information
- 4-6: Somewhat relevant, partially addresses the query
- 1-3: Marginally relevant
- 0: Not relevant

Respond with just the number."""
    
    def __init__(self, config: ProjectConfig, vector_store: VideoSegmentVectorStore):
        self.config = config
        self.vector_store = vector_store
        self.llm = None
        
    def retrieve_and_rank(self, query: str, 
                          top_k: int = 10,
                          rerank: bool = True) -> List[Tuple[InstructionalSegment, float]]:
        """
        Retrieve relevant segments and optionally rerank with LLM.
        """
        # Initial retrieval
        candidates = self.vector_store.search(query, top_k=top_k)
        
        if not rerank or self.llm is None:
            return candidates
            
        # LLM-based reranking
        reranked = []
        for segment, initial_score in candidates:
            # TODO: Call LLM for relevance scoring
            # relevance_score = self._get_llm_relevance(query, segment)
            relevance_score = initial_score  # Placeholder
            reranked.append((segment, relevance_score))
            
        # Sort by relevance score
        reranked.sort(key=lambda x: x[1], reverse=True)
        return reranked
    
    def process_query(self, query: str, 
                      error_code: Optional[str] = None,
                      machine_type: Optional[str] = None) -> Dict:
        """
        Process a maintenance query and return relevant segments.
        """
        # Enhance query with context
        enhanced_query = query
        if error_code:
            enhanced_query += f" Error code: {error_code}"
        if machine_type:
            enhanced_query += f" Machine: {machine_type}"
            
        # Retrieve and rank
        results = self.retrieve_and_rank(enhanced_query)
        
        # Filter by threshold
        filtered_results = [
            (seg, score) for seg, score in results 
            if score >= self.config.similarity_threshold
        ]
        
        return {
            'query': query,
            'enhanced_query': enhanced_query,
            'num_results': len(filtered_results),
            'segments': [
                {
                    'segment_id': seg.segment_id,
                    'video_id': seg.video_id,
                    'title': seg.title,
                    'summary': seg.summary,
                    'start_time': seg.start_time,
                    'end_time': seg.end_time,
                    'relevance_score': score
                }
                for seg, score in filtered_results
            ]
        }

print("RAG Pipeline class defined")

## 5.3 Goal 3 Evaluation

In [ ]:
class RetrievalEvaluator:
    """Evaluate retrieval performance."""
    
    @staticmethod
    def precision_at_k(retrieved: List[str], relevant: List[str], k: int) -> float:
        """Calculate Precision@K."""
        retrieved_k = retrieved[:k]
        relevant_set = set(relevant)
        hits = sum(1 for doc in retrieved_k if doc in relevant_set)
        return hits / k if k > 0 else 0
    
    @staticmethod
    def recall_at_k(retrieved: List[str], relevant: List[str], k: int) -> float:
        """Calculate Recall@K."""
        retrieved_k = set(retrieved[:k])
        relevant_set = set(relevant)
        hits = len(retrieved_k.intersection(relevant_set))
        return hits / len(relevant_set) if relevant_set else 0
    
    @staticmethod
    def mean_reciprocal_rank(retrieved: List[str], relevant: List[str]) -> float:
        """Calculate Mean Reciprocal Rank (MRR)."""
        relevant_set = set(relevant)
        for i, doc in enumerate(retrieved):
            if doc in relevant_set:
                return 1.0 / (i + 1)
        return 0.0
    
    @staticmethod
    def ndcg_at_k(retrieved: List[str], relevant: List[str], k: int) -> float:
        """Calculate Normalized Discounted Cumulative Gain (NDCG)@K."""
        relevant_set = set(relevant)
        
        # DCG
        dcg = 0.0
        for i, doc in enumerate(retrieved[:k]):
            if doc in relevant_set:
                dcg += 1.0 / np.log2(i + 2)  # +2 because i starts at 0
        
        # Ideal DCG
        ideal_dcg = sum(1.0 / np.log2(i + 2) for i in range(min(len(relevant_set), k)))
        
        return dcg / ideal_dcg if ideal_dcg > 0 else 0.0
    
    @staticmethod
    def evaluate_all(retrieved: List[str], relevant: List[str], k: int = 5) -> Dict:
        """Calculate all retrieval metrics."""
        return {
            f'precision@{k}': RetrievalEvaluator.precision_at_k(retrieved, relevant, k),
            f'recall@{k}': RetrievalEvaluator.recall_at_k(retrieved, relevant, k),
            'mrr': RetrievalEvaluator.mean_reciprocal_rank(retrieved, relevant),
            f'ndcg@{k}': RetrievalEvaluator.ndcg_at_k(retrieved, relevant, k)
        }

# Example evaluation
ret_evaluator = RetrievalEvaluator()

retrieved_docs = ['doc1', 'doc3', 'doc5', 'doc2', 'doc7']
relevant_docs = ['doc1', 'doc2', 'doc3', 'doc4']

metrics = ret_evaluator.evaluate_all(retrieved_docs, relevant_docs, k=5)
print("Retrieval Metrics:")
for metric, value in metrics.items():
    print(f"  {metric}: {value:.3f}")

---
# Phase 6: Micro-Clip Assembly & Safety Validation (Goal 4)
*Weeks 14-15: Video Assembly and Safety Compliance*

---

## 6.1 Video Clip Extraction

In [ ]:
class VideoClipExtractor:
    """
    Extracts video clips from source videos based on timestamps.
    """
    
    def __init__(self, config: ProjectConfig):
        self.config = config
        
    def extract_clip(self, video_path: Path, 
                     start_time: float, 
                     end_time: float,
                     output_path: Path) -> Path:
        """
        Extract a clip from source video.
        """
        # Using moviepy
        # from moviepy.editor import VideoFileClip
        # 
        # video = VideoFileClip(str(video_path))
        # clip = video.subclip(start_time, end_time)
        # clip.write_videofile(str(output_path), codec='libx264')
        # video.close()
        
        # Alternative: Using ffmpeg directly (more efficient)
        # import subprocess
        # duration = end_time - start_time
        # cmd = [
        #     'ffmpeg', '-ss', str(start_time),
        #     '-i', str(video_path),
        #     '-t', str(duration),
        #     '-c', 'copy',
        #     str(output_path)
        # ]
        # subprocess.run(cmd, check=True)
        
        logger.info(f"Extracted clip: {start_time:.2f}s - {end_time:.2f}s -> {output_path}")
        return output_path
    
    def concatenate_clips(self, clip_paths: List[Path], 
                          output_path: Path,
                          add_transitions: bool = True) -> Path:
        """
        Concatenate multiple clips into a single video.
        """
        # from moviepy.editor import VideoFileClip, concatenate_videoclips
        # 
        # clips = [VideoFileClip(str(p)) for p in clip_paths]
        # 
        # if add_transitions:
        #     # Add crossfade transitions
        #     final = concatenate_videoclips(clips, method='compose')
        # else:
        #     final = concatenate_videoclips(clips)
        # 
        # final.write_videofile(str(output_path), codec='libx264')
        # 
        # for clip in clips:
        #     clip.close()
        
        logger.info(f"Concatenated {len(clip_paths)} clips -> {output_path}")
        return output_path

extractor = VideoClipExtractor(config)
print("Video Clip Extractor initialized")

## 6.2 Micro-Clip Assembly Pipeline

In [ ]:
class MicroClipAssembler:
    """
    Assembles micro-clips from relevant segments.
    Handles sequencing, captions, and safety warnings.
    """
    
    def __init__(self, config: ProjectConfig):
        self.config = config
        self.extractor = VideoClipExtractor(config)
        
    def assemble_microclip(self, 
                           query: str,
                           segments: List[InstructionalSegment],
                           video_paths: Dict[str, Path]) -> MicroClip:
        """
        Assemble a micro-clip from relevant segments.
        
        Args:
            query: Original user query
            segments: List of relevant instructional segments
            video_paths: Mapping of video_id to video file path
        """
        clip_id = f"mc_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
        
        # Calculate total duration
        total_duration = sum(seg.end_time - seg.start_time for seg in segments)
        
        # Check duration constraints
        min_dur, max_dur = self.config.target_clip_duration
        if total_duration > max_dur:
            logger.warning(f"Total duration {total_duration:.0f}s exceeds max {max_dur}s")
            # TODO: Implement trimming logic
            
        # Generate captions
        captions = self._generate_captions(segments)
        
        # Collect safety warnings
        all_safety_warnings = []
        for seg in segments:
            all_safety_warnings.extend(seg.safety_warnings)
            
        micro_clip = MicroClip(
            clip_id=clip_id,
            query=query,
            source_segments=segments,
            total_duration=total_duration,
            captions=captions,
            safety_validated=False
        )
        
        logger.info(f"Assembled micro-clip {clip_id}: {len(segments)} segments, {total_duration:.0f}s")
        return micro_clip
    
    def _generate_captions(self, segments: List[InstructionalSegment]) -> List[Dict]:
        """Generate timed captions for the micro-clip."""
        captions = []
        current_time = 0.0
        
        for seg in segments:
            for transcript_seg in seg.transcript_segments:
                relative_start = current_time + (transcript_seg.start_time - seg.start_time)
                relative_end = current_time + (transcript_seg.end_time - seg.start_time)
                
                captions.append({
                    'text': transcript_seg.text,
                    'start': relative_start,
                    'end': relative_end
                })
                
            current_time += (seg.end_time - seg.start_time)
            
        return captions
    
    def render_microclip(self, micro_clip: MicroClip,
                         video_paths: Dict[str, Path],
                         add_captions: bool = True,
                         add_safety_intro: bool = True) -> Path:
        """
        Render the final micro-clip video file.
        """
        output_path = self.config.output_dir / f"{micro_clip.clip_id}.mp4"
        
        # TODO: Implement full rendering pipeline
        # 1. Extract individual clips
        # 2. Add captions/subtitles
        # 3. Add safety intro if needed
        # 4. Concatenate with transitions
        # 5. Add outro with metadata
        
        logger.info(f"Rendered micro-clip to {output_path}")
        micro_clip.output_path = output_path
        return output_path

assembler = MicroClipAssembler(config)
print("Micro-Clip Assembler initialized")

## 6.3 Safety Validation

In [ ]:
class SafetyValidator:
    """
    Validates micro-clips for safety compliance.
    Ensures all safety warnings are included and procedures are complete.
    """
    
    SAFETY_KEYWORDS = [
        'warning', 'caution', 'danger', 'safety', 'protective',
        'gloves', 'goggles', 'mask', 'ppe', 'lockout', 'tagout',
        'electrical', 'voltage', 'pressure', 'temperature', 'hazard'
    ]
    
    def __init__(self, config: ProjectConfig):
        self.config = config
        self.validation_log = []
        
    def validate(self, micro_clip: MicroClip) -> Tuple[bool, List[str]]:
        """
        Validate a micro-clip for safety compliance.
        
        Returns:
            Tuple of (is_valid, list of issues)
        """
        issues = []
        
        # Check 1: Safety warnings present
        all_warnings = []
        for seg in micro_clip.source_segments:
            all_warnings.extend(seg.safety_warnings)
            
        if not all_warnings:
            # Check transcript for safety-related content
            all_text = " ".join([
                ts.text.lower() 
                for seg in micro_clip.source_segments 
                for ts in seg.transcript_segments
            ])
            
            safety_mentioned = any(kw in all_text for kw in self.SAFETY_KEYWORDS)
            if safety_mentioned:
                issues.append("Safety content detected but no explicit warnings extracted")
                
        # Check 2: Clip duration within bounds
        min_dur, max_dur = self.config.target_clip_duration
        if micro_clip.total_duration > max_dur:
            issues.append(f"Duration {micro_clip.total_duration:.0f}s exceeds maximum {max_dur}s")
        if micro_clip.total_duration < min_dur:
            issues.append(f"Duration {micro_clip.total_duration:.0f}s below minimum {min_dur}s")
            
        # Check 3: Captions present
        if not micro_clip.captions:
            issues.append("No captions generated")
            
        # Check 4: Segment continuity
        # Ensure segments don't have large gaps that might indicate missing steps
        # TODO: Implement continuity check
        
        is_valid = len(issues) == 0
        
        # Log validation
        self.validation_log.append({
            'clip_id': micro_clip.clip_id,
            'timestamp': datetime.now().isoformat(),
            'is_valid': is_valid,
            'issues': issues
        })
        
        return is_valid, issues
    
    def mark_validated(self, micro_clip: MicroClip, 
                       reviewer: str,
                       notes: str = "") -> None:
        """Mark a micro-clip as validated by a human reviewer."""
        micro_clip.safety_validated = True
        micro_clip.validation_notes = f"Validated by {reviewer}: {notes}"
        
        self.validation_log.append({
            'clip_id': micro_clip.clip_id,
            'timestamp': datetime.now().isoformat(),
            'action': 'manual_validation',
            'reviewer': reviewer,
            'notes': notes
        })
        
        logger.info(f"Micro-clip {micro_clip.clip_id} validated by {reviewer}")

validator = SafetyValidator(config)
print("Safety Validator initialized")

---
# Phase 7: Evaluation & Iteration
*Weeks 16-17: System Evaluation and Refinement*

---

## 7.1 End-to-End Evaluation Pipeline

In [ ]:
class SystemEvaluator:
    """
    Comprehensive evaluation of the entire pipeline.
    """
    
    def __init__(self, config: ProjectConfig):
        self.config = config
        self.results = {}
        
    def evaluate_pipeline(self, test_queries: List[Dict]) -> pd.DataFrame:
        """
        Run end-to-end evaluation on test queries.
        
        Args:
            test_queries: List of dicts with 'query', 'expected_videos', etc.
        """
        results = []
        
        for test_case in test_queries:
            query = test_case['query']
            expected = test_case.get('expected_segment_ids', [])
            
            # TODO: Run full pipeline and collect metrics
            # retrieved = rag_pipeline.process_query(query)
            # micro_clip = assembler.assemble_microclip(...)
            # is_valid, issues = validator.validate(micro_clip)
            
            result = {
                'query': query,
                'num_expected': len(expected),
                'num_retrieved': 0,  # Placeholder
                'precision': 0.0,
                'recall': 0.0,
                'clip_duration': 0.0,
                'safety_validated': False
            }
            results.append(result)
            
        return pd.DataFrame(results)
    
    def generate_report(self, results_df: pd.DataFrame) -> str:
        """Generate evaluation report."""
        report = []
        report.append("# AI Content Development - Evaluation Report")
        report.append(f"Generated: {datetime.now().isoformat()}\n")
        
        report.append("## Summary Statistics")
        report.append(f"- Total test cases: {len(results_df)}")
        report.append(f"- Average Precision: {results_df['precision'].mean():.3f}")
        report.append(f"- Average Recall: {results_df['recall'].mean():.3f}")
        report.append(f"- Safety Validation Rate: {results_df['safety_validated'].mean():.1%}\n")
        
        return "\n".join(report)

evaluator = SystemEvaluator(config)
print("System Evaluator initialized")

## 7.2 Visualization and Analysis

In [ ]:
def plot_evaluation_results(results_df: pd.DataFrame):
    """Create visualizations of evaluation results."""
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    
    # Precision distribution
    axes[0, 0].hist(results_df['precision'], bins=20, edgecolor='black')
    axes[0, 0].set_xlabel('Precision')
    axes[0, 0].set_ylabel('Count')
    axes[0, 0].set_title('Precision Distribution')
    axes[0, 0].axvline(x=0.8, color='r', linestyle='--', label='Target (0.8)')
    axes[0, 0].legend()
    
    # Recall distribution
    axes[0, 1].hist(results_df['recall'], bins=20, edgecolor='black')
    axes[0, 1].set_xlabel('Recall')
    axes[0, 1].set_ylabel('Count')
    axes[0, 1].set_title('Recall Distribution')
    axes[0, 1].axvline(x=0.7, color='r', linestyle='--', label='Target (0.7)')
    axes[0, 1].legend()
    
    # Clip duration distribution
    axes[1, 0].hist(results_df['clip_duration'], bins=20, edgecolor='black')
    axes[1, 0].set_xlabel('Duration (seconds)')
    axes[1, 0].set_ylabel('Count')
    axes[1, 0].set_title('Micro-Clip Duration Distribution')
    axes[1, 0].axvline(x=60, color='g', linestyle='--', label='Min (60s)')
    axes[1, 0].axvline(x=120, color='r', linestyle='--', label='Max (120s)')
    axes[1, 0].legend()
    
    # Precision vs Recall scatter
    axes[1, 1].scatter(results_df['precision'], results_df['recall'], alpha=0.6)
    axes[1, 1].set_xlabel('Precision')
    axes[1, 1].set_ylabel('Recall')
    axes[1, 1].set_title('Precision vs Recall')
    axes[1, 1].axhline(y=0.7, color='r', linestyle='--', alpha=0.5)
    axes[1, 1].axvline(x=0.8, color='r', linestyle='--', alpha=0.5)
    
    plt.tight_layout()
    plt.savefig(config.output_dir / 'evaluation_results.png', dpi=150)
    plt.show()
    
# Example with dummy data
dummy_results = pd.DataFrame({
    'precision': np.random.uniform(0.6, 1.0, 50),
    'recall': np.random.uniform(0.5, 0.9, 50),
    'clip_duration': np.random.uniform(45, 150, 50)
})

print("Evaluation visualization function defined")
# Uncomment to generate plots with real data:
# plot_evaluation_results(dummy_results)

---
# Phase 8: Prototype Interface
*Week 18: Demo Interface and Final Delivery*

---

## 8.1 Gradio Demo Interface

In [ ]:
def create_demo_interface():
    """
    Create a Gradio demo interface for the micro-clip retrieval system.
    """
    import gradio as gr
    
    def search_and_generate(query: str, error_code: str, machine_type: str):
        """
        Search for relevant segments and generate micro-clip.
        """
        # TODO: Connect to actual pipeline
        # results = rag_pipeline.process_query(query, error_code, machine_type)
        # micro_clip = assembler.assemble_microclip(...)
        
        # Placeholder response
        response = f"""
        **Query:** {query}
        **Error Code:** {error_code or 'Not specified'}
        **Machine Type:** {machine_type or 'Not specified'}
        
        **Results:**
        - Found 3 relevant video segments
        - Estimated clip duration: 90 seconds
        - Safety warnings: 2 included
        
        *Demo mode - actual video generation not implemented*
        """
        return response, None  # response text, video file
    
    # Create interface
    interface = gr.Interface(
        fn=search_and_generate,
        inputs=[
            gr.Textbox(
                label="Maintenance Query",
                placeholder="e.g., How do I replace the filter in module X?"
            ),
            gr.Textbox(
                label="Error Code (optional)",
                placeholder="e.g., E-4521"
            ),
            gr.Dropdown(
                label="Machine Type",
                choices=["EUV System", "DUV System", "Metrology Tool", "Other"],
                value="EUV System"
            )
        ],
        outputs=[
            gr.Markdown(label="Search Results"),
            gr.Video(label="Generated Micro-Clip")
        ],
        title="ASML AI Content Development - Micro-Clip Generator",
        description="""Search through training videos to find relevant troubleshooting 
        instructions and automatically generate short micro-clips.""",
        examples=[
            ["How do I calibrate the alignment sensor?", "E-1234", "EUV System"],
            ["Steps to replace the vacuum pump", "", "DUV System"],
            ["Troubleshooting wafer handling errors", "W-5678", "EUV System"]
        ]
    )
    
    return interface

# Create and launch interface
# Uncomment to launch:
# demo = create_demo_interface()
# demo.launch(share=True)

print("Demo interface function defined")
print("To launch: demo = create_demo_interface(); demo.launch()")

## 8.2 Complete Pipeline Orchestration

In [ ]:
class MicroLearningPipeline:
    """
    Complete end-to-end pipeline orchestrator.
    Coordinates all components for video processing and micro-clip generation.
    """
    
    def __init__(self, config: ProjectConfig):
        self.config = config
        
        # Initialize components
        self.data_loader = VideoDataLoader(config)
        self.asr_pipeline = ASRPipeline(config)
        self.structurer = TranscriptStructurer(config)
        self.aligner = TimestampAligner(config)
        self.segmenter = SemanticSegmenter(config)
        # self.vector_store = VideoSegmentVectorStore(config)
        # self.rag_pipeline = MaintenanceRAGPipeline(config, self.vector_store)
        self.assembler = MicroClipAssembler(config)
        self.validator = SafetyValidator(config)
        
        logger.info("Pipeline initialized")
        
    def process_video(self, video_path: Path) -> List[InstructionalSegment]:
        """
        Process a single video through the full pipeline.
        """
        video_id = video_path.stem
        logger.info(f"Processing video: {video_id}")
        
        # Step 1: Transcribe
        transcript_segments = self.asr_pipeline.transcribe(video_path)
        self.asr_pipeline.save_transcript(transcript_segments, video_id)
        
        # Step 2: Refine timestamps
        refined_segments = self.aligner.refine_timestamps(transcript_segments, video_path)
        
        # Step 3: Semantic segmentation
        instructional_segments = self.segmenter.create_semantic_segments(
            refined_segments, video_id
        )
        
        # Step 4: LLM structuring (title, summary, steps extraction)
        # TODO: Enhance segments with LLM-generated content
        
        logger.info(f"Processed {video_id}: {len(instructional_segments)} segments")
        return instructional_segments
    
    def generate_microclip(self, query: str, 
                           error_code: Optional[str] = None) -> MicroClip:
        """
        Generate a micro-clip for a given query.
        """
        # Step 1: Retrieve relevant segments
        # results = self.rag_pipeline.process_query(query, error_code)
        # segments = [r['segment'] for r in results['segments']]
        
        # Placeholder
        segments = []
        
        # Step 2: Assemble micro-clip
        micro_clip = self.assembler.assemble_microclip(
            query=query,
            segments=segments,
            video_paths={}  # TODO: Provide actual video paths
        )
        
        # Step 3: Validate
        is_valid, issues = self.validator.validate(micro_clip)
        if not is_valid:
            logger.warning(f"Validation issues: {issues}")
            
        return micro_clip
    
    def run_batch_processing(self):
        """
        Process all videos in the raw videos directory.
        """
        videos = self.data_loader.scan_videos()
        
        all_segments = []
        for video in videos:
            try:
                segments = self.process_video(video.filepath)
                all_segments.extend(segments)
            except Exception as e:
                logger.error(f"Error processing {video.video_id}: {e}")
                
        # Add to vector store
        # self.vector_store.add_segments(all_segments)
        # self.vector_store.save(self.config.embeddings_dir)
        
        logger.info(f"Batch processing complete: {len(all_segments)} total segments")
        return all_segments

# Initialize complete pipeline
pipeline = MicroLearningPipeline(config)
print("Complete pipeline initialized and ready!")

---
# Project Documentation & Notes

---

## Research Question

**Main RQ:** How can an AI-powered system effectively extract, structure, and retrieve relevant instructional segments from existing training videos to automatically assemble accurate, safety-compliant micro-clips for machine maintenance troubleshooting?

### Sub-Questions
1. How accurately can ASR and LLM-based structuring extract and organize maintenance instructions from training videos?
2. How effectively can semantic segmentation identify and timestamp relevant instructional segments?
3. What retrieval approaches achieve the highest relevance for error-driven maintenance queries?
4. How can safety warnings and metadata be integrated to ensure compliant micro-clip outputs?

## Progress Tracking

Use this section to track progress through the project phases.

In [ ]:
# Project progress tracker
progress = pd.DataFrame({
    'Phase': [
        'Phase 1: Project Setup',
        'Phase 2: Literature Review',
        'Phase 3: Data Exploration & ASR',
        'Phase 4: Timestamp Alignment',
        'Phase 5: Retrieval System',
        'Phase 6: Micro-Clip Assembly',
        'Phase 7: Evaluation',
        'Phase 8: Final Delivery'
    ],
    'Weeks': ['1-2', '3-4', '5-7', '8-10', '11-13', '14-15', '16-17', '18'],
    'Goal': ['Setup', 'Research', 'Goal 1', 'Goal 2', 'Goal 3', 'Goal 4', 'Evaluation', 'Delivery'],
    'Status': ['Not Started'] * 8,
    'Notes': [''] * 8
})

progress

## Risk Register

Track identified risks and mitigation strategies.

In [ ]:
risks = pd.DataFrame({
    'Risk': [
        'Data quality issues in training videos',
        'ASR accuracy below threshold',
        'LLM hallucinations in structuring',
        'Insufficient compute resources',
        'Safety validation delays'
    ],
    'Likelihood': ['Medium', 'Low', 'Medium', 'Low', 'Medium'],
    'Impact': ['High', 'High', 'Medium', 'Medium', 'High'],
    'Mitigation': [
        'Early data exploration, data cleaning pipeline',
        'Use larger Whisper model, human-in-loop correction',
        'Prompt engineering, validation against source',
        'Cloud GPU resources, model optimization',
        'Early engagement with safety reviewers'
    ],
    'Status': ['Monitoring'] * 5
})

risks

---
## Next Steps

1. **Add training video data** to `./data/raw_videos/`
2. **Configure LLM access** (API key or local model)
3. **Run initial transcription** on sample videos
4. **Evaluate ASR quality** against manual transcripts
5. **Iterate on segmentation** based on stakeholder feedback

---

*End of Notebook Template*